# Reweighting a MELD run: weighted ensembles and permutation vectors

This notebook uses the **reweightmeld** package (`reweightmeld/src/reweightmeld` in project 14). It needs neither MELD, OpenMM
nor MELDDiffuse: it only reads the MELD data store (`Data/Blocks/block_*.nc`, `Data/trajectory.pdb`).

It produces:
1. **Permutation vectors and walker ids**: which walker (continuous trajectory) sits in which replica slot at every stage.
2. **MBAR weights** of every (stage, slot) frame in the target state, slot 0 (lowest temperature, full restraints), with diagnostics; optionally BayesMBAR posterior weights.
3. Optionally, **per-adaptation-step reweighting**: each constant-alpha segment is reweighted to its own slot 0, then the segments are pooled.
4. Optionally, a **weighted resample** of structures as an all-atom DCD/PDB.
5. **Diagnostic and structure-analysis plots**, shown here and saved as PNG.

The same steps run from the command line: `reweightmeld run RUN_DIR -o PREFIX --bayes --resample 1000 --report` (see `README.md`).

**How to use:** run section 0 once if reweightmeld is not installed, edit the parameters in section 1, then run all cells.

## 0. Installation (run once, only if needed)

The next cell checks whether `reweightmeld` can be imported. If not, it installs it with pip from `PACKAGE`: this folder
(editable install) or a git URL. Set `INSTALL_BAYES = True` to also install BayesMBAR (posterior uncertainties of the
weights; needs JAX, CPU is enough). Restart the kernel after installing.

On HiPerGator the ready-made environment is `reweightmeld_env` (kernel "Python (reweightmeld_env)").

In [ ]:
INSTALL_BAYES = True
PACKAGE = "/orange/alberto.perezant/imesh.ranaweera/projects/14_reweight_meld_trajectories/reweightmeld"   # or "git+https://github.com/<user>/reweightmeld.git"

import importlib.util, subprocess, sys
from pathlib import Path

if importlib.util.find_spec("reweightmeld") is None:
    extra = "[bayes]" if INSTALL_BAYES else ""
    if Path(PACKAGE).is_dir():
        args = ["-e", PACKAGE + extra]
    else:
        args = [f"reweightmeld{extra} @ {PACKAGE}"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
    print("Installed. Restart the kernel, then continue from section 1.")
else:
    print("reweightmeld is already installed:", importlib.util.find_spec("reweightmeld").origin)

## 1. Parameters (edit this cell)

| Parameter | Meaning |
|---|---|
| `RUN_DIR` | MELD run directory, containing `Data/Blocks/block_*.nc` and `Data/trajectory.pdb` |
| `OUT_PREFIX` | prefix of all output files |
| `START_STAGE` | earliest stage to use. 201 skips the restraint ramp of the usual setup scripts; use 1 if your run has no ramp |
| `TARGET_SLOT` | ladder slot to reweight to (0 = lowest temperature, full restraints) |
| `USE_SEGMENTS` | `False`: only stages after the last alpha adaptation. `True`: every constant-alpha segment, each reweighted to its own slot 0, then pooled |
| `MIN_SEGMENT_LEN` | segments shorter than this (in stages) are skipped |
| `USE_BAYES` | also compute BayesMBAR weights (only if BayesMBAR is installed) |
| `N_RESAMPLE` | number of structures in the weighted resample (0 = skip) |
| `SEED` | random seed of the resample |
| `N_ANALYSIS` | structures drawn by MBAR weight for the analysis plots of section 9  |
| `N_STATES` | number of conformational states (k-means on the Cα-distance PCA) |
| `RESTRAINT_FILES` | optional MELD restraint files for the satisfaction plots, e.g. `[("NOE.dat", "noe"), ("local_NOE.dat", "noe"), ("hydrophobe.dat", "contact", 5.0)]` (paths relative to `RUN_DIR`; the third value is the upper bound in Å for files without a distance column; `[]` = skip) |

In [ ]:
RUN_DIR = "/path/to/meld_run"
OUT_PREFIX = "out/my_system"
START_STAGE = 201
TARGET_SLOT = 0
USE_SEGMENTS = False
MIN_SEGMENT_LEN = 50
USE_BAYES = True
N_RESAMPLE = 1000
SEED = 0
N_ANALYSIS = 2000
N_STATES = 5
RESTRAINT_FILES = []

## 2. Imports and opening the run

In [ ]:
%matplotlib inline
import json

import numpy as np
from reweightmeld import MeldRun, Reweighting, reweight, report
from reweightmeld.mbar import bayes_available
from reweightmeld.windows import alpha_change_stages, constant_alpha_segments, frozen_window

PLOT_DIR = OUT_PREFIX + "_plots"
run = MeldRun(RUN_DIR)
print(run)
print("BayesMBAR available:", bayes_available())

## 3. Choose the stage window(s)

MBAR needs every replica slot to be a fixed thermodynamic state, so the alphas (the ladder) must stay constant over a window.

- **Single window** (`USE_SEGMENTS = False`): from the stage after the last alpha change (or `START_STAGE`, whichever is later) to the end.
- **Segments** (`USE_SEGMENTS = True`): the run is split wherever the alphas change. The first stage after each change is dropped, as in MELDDiffuse.

In [ ]:
segments = (constant_alpha_segments(run, START_STAGE, MIN_SEGMENT_LEN) if USE_SEGMENTS
            else [frozen_window(run, START_STAGE)])
print("ladder changed at stages:", alpha_change_stages(run).tolist())
for i, seg in enumerate(segments):
    print(f"segment {i}: stages {seg[0]}..{seg[-1]} ({len(seg)} stages), "
          f"alphas = {np.round(run.alphas([seg[0]])[0], 3)}")

## 4. MBAR reweighting of each window, and combining them

For every window, `reweight` builds the reduced-energy matrix from MELD's `energy_matrix` (kT, includes the prior of
MELD-adapt's discrete parameters), runs MBAR (and BayesMBAR with `USE_BAYES`), and returns the weight of every
(stage, slot) frame in `TARGET_SLOT`. It also follows the permutation vectors to get the walker ids.

**Diagnostics to check:**
- `ess`: effective sample size, corrected for correlation along walkers.
- `u_target_reweighted` ≈ `u_target_direct`: the reweighted mean energy should match the plain average in the target slot.
- `min_neighbor_overlap`: MBAR overlap between neighbouring slots; very small values (< ~0.03) mean poorly connected states.
- `n_walkers_in_target`: how many different walkers visited the target slot.

**Several segments** (`USE_SEGMENTS = True`) are each an estimate of the same target distribution **only if slot 0 is
the same physical state in every segment**. Pooling is refused if the target alpha differs. It cannot check whether
the prior on the discrete restraint parameters changed between adaptation steps: if `u_target_direct` jumps between
segments in the table below, keep the segments separate. Early segments may also not be equilibrated; drop them with
`START_STAGE` if they look different. Segments are mixed in proportion to their effective sample size.

In [ ]:
rw = reweight(run, target=TARGET_SLOT, start_stage=START_STAGE, segments=USE_SEGMENTS,
              min_segment_len=MIN_SEGMENT_LEN, bayes=USE_BAYES, seed=SEED)
for f in rw.save(OUT_PREFIX):
    print("saved", f)
summary = rw.summary()
print(json.dumps(summary["windows"], indent=1))
print(f"{rw.n_frames} frames, sum of weights = {rw.weight.sum():.6f}, "
      f"pooled Kish ESS (uncorrected) = {summary['kish_ess_pooled']:.0f}, BayesMBAR weights: {summary['bayes']}")

## 5. Permutation vectors and walker ids

MELD stores one permutation vector per stage. Following them gives the **walker id** in every slot:
`rw.walker_ids[s, j]` is the walker sitting in slot `j` at stage `rw.perm_stages[s]`. Walkers are followed from
stage 1 (stage 0 is the starting state: every walker in its own slot, no permutation stored), through every stage
(including those dropped between segments), so walker ids are the same whatever `START_STAGE` is. They are saved in
`*.perm.npz`, where the `used` mask marks the stages inside a reweighted window.

In [ ]:
print("permutation vectors:", rw.perms.shape, " stages", rw.perm_stages[0], "..", rw.perm_stages[-1])
print("walker ids of the first 3 stages:\n", rw.walker_ids[:3])

## 6. Weighted resample of structures (optional)

Draws `N_RESAMPLE` frames with probability proportional to their weight and writes them as an all-atom trajectory, using frame 1 of `Data/trajectory.pdb` as the topology. `*.ensemble_index.npy` gives the row of `reweight.npz` each frame came from. `rw.resample(run, n, seed)` returns the same as an mdtraj trajectory without writing files.

In [ ]:
if N_RESAMPLE:
    ens, idx = rw.save_ensemble(run, OUT_PREFIX, N_RESAMPLE, SEED)
    print(f"saved {N_RESAMPLE} frames to {OUT_PREFIX}.ensemble.dcd (+ .pdb topology)")
    print("unique frames drawn:", len(np.unique(idx)))

## 7. Reloading saved results (optional)

Everything above can be reloaded without recomputing the weights:

```python
rw = Reweighting.load(OUT_PREFIX)
run = MeldRun(rw.run_dir)
report.diagnostics(run, rw, PLOT_DIR, show=True)
```

## 8. Reweighting diagnostics (plots)

`report.diagnostics` makes every plot, shows it in the notebook and saves it as a PNG to `<OUT_PREFIX>_plots/`.

| Plot | What to look for |
|---|---|
| 8a Ladder and window | alphas stop changing before the reweighted window starts (shaded) |
| 8b Where the target weight comes from | most weight from the target slot and its neighbours; a large share from distant hot slots means poor overlap |
| 8c Neighbour overlap | every neighbouring pair well above ~0.03; a dip is a bottleneck in the ladder |
| 8d Walkers | several walkers visit the target slot (one walker = one trajectory, so few walkers = few independent samples) |
| 8e Weight concentration | how many frames carry 50 / 90 / 99.9 % of the weight; Kish ESS vs walker-corrected ESS |
| 8f Weight over time | roughly flat; a trend means the window is not equilibrated |
| 8g BayesMBAR (if computed) | relative uncertainty of the frame weights |

A walker is one continuous trajectory moving up and down the ladder. Frames of the same walker are correlated, so the
number of different walkers that reach the target slot, and how long each one stays, sets how many independent samples
the reweighted ensemble really has. In 8d, the right panel shows each walker's statistical inefficiency g of the target-state
energy (the walker-corrected ESS is the Kish ESS divided by the median g).

In [ ]:
diag_files = report.diagnostics(run, rw, PLOT_DIR, show=True)

## 9. Structure analysis of the reweighted ensemble (plots)

`N_ANALYSIS` structures are drawn with probability equal to their MBAR weight. For comparison, two **unweighted** sets of the same size are drawn from the same window: the target slot alone
(what you get without reweighting) and all slots pooled (what the reweighting removes).

- **9a** Cα-distance PCA: every structure is described by its Cα–Cα distances (|i−j| ≥ 3), so no superposition is needed.
  The axes are fitted on the reweighted ensemble; the other sets are projected onto them. States = k-means
  (`N_STATES`) on the first 10 PCs; the structure closest to each state centre is saved as `<OUT_PREFIX>.state<k>.pdb`.
- **9b** state populations, **9c** Cα RMSF and radius of gyration, **9d** Cα contact probabilities (< 8 Å),
  **9e** restraint satisfaction (only with `RESTRAINT_FILES`; MELD's all-atom frames already carry hydrogens, so the
  restraint atom names are used as they are; a group counts as satisfied when any of its lines is within its bound).

In [ ]:
res = report.structure(run, rw, PLOT_DIR, prefix=OUT_PREFIX, n_analysis=N_ANALYSIS, n_states=N_STATES,
                       restraints=RESTRAINT_FILES, seed=SEED, show=True)
print("state populations:")
for k, pop in res["states"]["populations"].items():
    print(f"  {k:30s}", np.round(pop, 3))

## 10. Output files

| File | Content |
|---|---|
| `*.perm.npz` | `stage`, `permutation_vectors` (S, R), `walker_ids` (S, R), `used` (stage inside a reweighted window) |
| `*.summary.json` | settings and per-window diagnostics |
| `*.reweight.npz` | `stage`, `slot`, `segment`, `weight` of every frame (sums to 1), `segment_mix`; `weight_bayes(_std)` with BayesMBAR |
| `*.ensemble.dcd/.pdb` | weighted resample of structures |
| `*.ensemble_index.npy` | row of `reweight.npz` behind each resampled frame |
| `*_plots/*.png` | the diagnostic (section 8) and structure (section 9) plots |
| `*.state<k>.pdb` | representative structure of each conformational state (section 9) |

To load a frame yourself: `run.positions([stage], [slot])` returns its coordinates in nm; `run.trajectory(stages, slots)` returns an mdtraj trajectory.